In [42]:
FILE_PATH = 'C://Users//User//Downloads//bitmex_data_1m.csv'
#FILE_PATH = 'D://bitmex_data_1m.csv'
JPG_DIRECTORY = 'C://Test'

from abc import ABC, abstractmethod
from common import *
import plotly
import pandas as pd

import warnings
warnings.simplefilter(action='ignore', category=Warning)

In [4]:
#import sys
#!{sys.executable} -m pip install twilio

In [5]:
#import sys
#!{sys.executable} -m pip install -U kaleido

In [6]:
#test_df = pd.read_csv(FILE_PATH, delimiter=',')
#test_df

In [7]:
import numpy as np

In [8]:


# 데이터 로드 전략 인터페이스
class DataLoaderStrategy(ABC):
    @abstractmethod
    def load_data(self):
        #1분봉 데이터를 불러오는 과정
        pass
    
    #@abstractmethod
    #def pre_precessing(self, df, base_delta, from_date=None):
    #    pass
    #    #1분봉 데이터를 여러 분봉으로 변경하는 함수
        

# 구체적인 데이터 로드 전략: CSV 로드
class BitmexCSVDataLoader(DataLoaderStrategy):
    
    raw_data = None #1분봉 df를 담을 변수
    final_df = None
    
    def __init__(self, base_delta: int, from_date:str = None):
        self.base_delta = base_delta  # 인스턴스 변수로 저장
        self.from_date = from_date
    
    def load_data(self):
        print("CSV 데이터를 로드하고 Nan을 제거합니다.")
        import pandas as pd
        df = pd.read_csv(FILE_PATH, delimiter=',')
        df = df.dropna()
        
        print(f"입력받은 {self.base_delta}분봉으로 {self.from_date} 부터 표현합니다.")
        
        if(self.from_date):
            df = df.loc[df.timestamp >= self.from_date]  #일단 GMT 니까 1분 뒤로 조정할 걸 생각하고 +1분부터 가져오면 된다.
        
        print("CSV 데이터 로드 완료.")
        
        df=df[['timestamp','high','low','open','close']]
        
        #timestamp 를 kst 로 조정
        df['timestamp_kst'] = df['timestamp'].apply(convert_gmt_to_kst)
        
        #클래스에 세팅
        raw_data = df
        
        df = self.__pre_precessing(df, self.base_delta, self.from_date)
        print(f"전처리 완료")

        return df
    
    def __pre_precessing(self, df, base_delta , from_date=None):
        df = df[['timestamp_kst','open','low','high','close']]
    
        df = df.reset_index()
        
        #int 형태의 timestamp 열도 추가
        df['timestamp_int'] = df['timestamp_kst'].apply(convert_to_timestamp)
        
        #다시 필요한 컬럼만 정돈
        df = df[['timestamp_kst','timestamp_int','open', 'low','high','close']]
        
        #최종 x분봉의 형태구현 base_delta = 15, 45, 240, 1440
        final_df = df[['low']].rolling(window=base_delta).min()     
        final_df['open'] = df[['open']].rolling(window=base_delta).apply(lambda x: x[0], raw=True)
        final_df['high'] = df[['high']].rolling(window=base_delta).max() 
        final_df['close'] = df['close']                         
        final_df['timestamp_int'] = df[['timestamp_int']]-(base_delta*60-60)  
        
        final_df = final_df[base_delta-1:] #window수 -1 값만큼 버리고
        final_df['timestamp_kst'] = final_df['timestamp_int'].apply(convert_timestamp_to_datetime_str)
        final_df = final_df.reset_index()[['timestamp_kst','open','low','high','close','timestamp_int']] #리셋재구성
        
        final_df = final_df[final_df['timestamp_int']%(base_delta*60)==0]
        
        return final_df
        

# 데이터 처리 전략 인터페이스
class DataProcessingStrategy(ABC):
    @abstractmethod
    def process_data(self, data):
        pass

# 구체적인 데이터 처리 전략: 이동 평균 계산
class MovingAverageProcessing(DataProcessingStrategy):
    def process_data(self, data):
        print("이동 평균을 계산했습니다.")
        data['ma_20']=self.__sma(data,20)
        data['ma_60']=self.__sma(data,60)
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __sma(self, data, period=20):
        return data['close'].rolling(window=period, min_periods=1).mean()
    
# 구체적인 데이터 처리 전략: RSI 계산
class RSIProcessing(DataProcessingStrategy):
    def process_data(self, data):
        data['RSI'] = self.__rsi(data)
        print("RSI를 계산했습니다.")
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __rsi(self, data, period=14):
    
        import numpy as np
        
        delta = data['close'].diff(1)  # 종가의 변화량 계산
        gain = np.where(delta > 0, delta, 0)  # 상승분
        loss = np.where(delta < 0, -delta, 0)  # 하락분
    
        avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
        avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()
        
        rs = avg_gain / (avg_loss + 1e-10)  # 0으로 나누는 오류 방지
        rsi = 100 - (100 / (1 + rs))
        
        rsi.index = data.index
        
        return rsi

#고점을 찾는 처리 전략
class HighPointScoringProcessing(DataProcessingStrategy):
    '''
    메인 df에 'high_score' 라는 컬럼을 추가하고, 외부 변수의 리스트로 들어온 
    밴드 값을 shift 하면서 그중에 가장 큰 가격에 +1 스코어를 한다.
    '''
    
    
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
    
    def process_data(self, data):
        data = self.__get_high_score_by_list(data,self.bandwith_list)
        return data
    
    def __get_high_score_by_list(self, origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_high_score_with_bandwidth(origin_df, 'high', i)
                else:
                    return_df = self.__get_high_score_with_bandwidth(return_df, 'high', i, reset=False)
    
        return return_df
    
    
        
    def __get_high_score_with_bandwidth(self, target_df, column_name, bandwidth, reset=True):
        '''
        df를 제공하면서 밴드 값을 같이 제공하면 이를 반복문으로 돌아가면서 score 를 쌓는 함수
        '''
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['high_score']=0
        #end_index = len(target_df) - bandwidth
        #for idx,i in enumerate(range(end_index+1)): #이렇게 하니까 맨 뒷쪽 점수가 부여될 수가 없더라
        for idx,i in enumerate(range(len(target_df))):
            
            max_index = target_df.iloc[i:i+bandwidth][column_name].idxmax()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
        #
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
        #
            #print(f"체크값 :{target_df.loc[max_index]['high_score']+1}")
            #
            target_df.loc[max_index,'high_score'] = target_df.loc[max_index]['high_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
        
        print(f"가장 높은 점수:{target_df.loc[target_df['high_score'].idxmax()]['high_score']}")
        
        return target_df

#저점을 찾는 처리 전략
class LowPointScoringProcessing(DataProcessingStrategy):
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
        
    def process_data(self, data):
        data = self.__get_low_score_by_list(data, self.bandwith_list)
        return data
    
    def __get_low_score_by_list(self,origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_low_score_with_bandwidth(origin_df, 'low', i)
                else:
                    return_df = self.__get_low_score_with_bandwidth(return_df, 'low', i, reset=False)
        
        return return_df
    
    
    def __get_low_score_with_bandwidth(self,target_df, column_name, bandwidth, reset=True):
    
    
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['low_score']=0
        #end_index = len(target_df) - bandwidth
        #for idx,i in enumerate(range(end_index+1)):
        for idx,i in enumerate(range(len(target_df))):
            
            min_index = target_df.iloc[i:i+bandwidth][column_name].idxmin()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
            
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
            
            #print(f"체크값 :{target_df.loc[max_row_index]['high_score']+1}")
            
            target_df.loc[min_index,'low_score'] = target_df.loc[min_index]['low_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
            
        print(f"가장 높은 점수:{target_df.loc[target_df['low_score'].idxmax()]['low_score']}")

        return target_df

# 고점만을 필터팅하는 처리 전략
class GetHighPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['high_score']!=0]
        final_high_point=self.__apply_threshold_with_normalizing_for_high_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_high_point
        
    #df와 타겟 컬럼명을 받아서 해당 %이상의 값만 필터링하는 함수
    def __apply_threshold_with_normalizing_for_high_value(self, target_df, target_column, threshold):
    
        #min-max 정규화
        target_df['normalized_value_high'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_high'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #상위 5%에 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_high'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'origin_high_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'high_for_graph'] = df_calculated['high']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_high'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_high_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['high_score'] > base_score]
        df_calculated.loc[df_calculated['high_score'] > base_score, 'origin_score'] = df_calculated['high_score']
        df_calculated.loc[df_calculated['high_score'] > base_score, 'high_score'] = df_calculated['high']
    
        import numpy as np
        df_calculated.loc[df_calculated['high_score'] <= base_score, 'high_score'] = np.nan
        
        return df_calculated
    
# 저점만을 필터팅하는 처리 전략
class GetLowPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['low_score']!=0]
        final_low_point=self.__apply_threshold_with_normalizing_for_low_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_low_point
    
    def __apply_threshold_with_normalizing_for_low_value(self, target_df, target_column, threshold):
        #min-max 정규화
        target_df['normalized_value_low'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_low'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #threshold 로 적은 수 이하로 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_low'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_low'] >= threshold, 'origin_low_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_low'] >= threshold, 'low_for_graph'] = df_calculated['low']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_low'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_low_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['low_score'] > base_score]
        df_calculated.loc[df_calculated['low_score'] > base_score, 'origin_score'] = df_calculated['low_score']
        df_calculated.loc[df_calculated['low_score'] > base_score, 'low_score'] = df_calculated['low']
    
        import numpy as np
        df_calculated.loc[df_calculated['low_score'] <= base_score, 'low_score'] = np.nan
        
        return df_calculated
    
# 고점/저점이 오르/내리는 구간을 구하는 처리 전략
class GetTrendSections(DataProcessingStrategy):
    
    def __init__(self, direction, target_column, minimum_interval, tolerate_interval):
        self.direction = direction
        self.target_column = target_column
        self.minimum_interval = minimum_interval
        self.tolerate_interval = tolerate_interval
        
    def process_data(self, data): 
        self.resource_df = data
        
        if(self.direction=='high'): #고점일 경우
            #이전 값보다 높거나 같으면 True찍기
            self.resource_df['increasing'] = self.resource_df['high_for_graph'] >= self.resource_df['high_for_graph'].shift()
        
            #이전 값보다 낮거나 같으면 Ture찍기
            self.resource_df['decreasing'] = self.resource_df['high_for_graph'] <= self.resource_df['high_for_graph'].shift()
            
        else:#저점일 경우
            #이전 값보다 높높거나 같으면으면 True찍기
            self.resource_df['increasing'] = self.resource_df['low_for_graph'] >= self.resource_df['low_for_graph'].shift()
            
            #이전 값보다 낮거나 같으면으면 True찍기
            self.resource_df['decreasing'] = self.resource_df['low_for_graph'] <= self.resource_df['low_for_graph'].shift()
        
        #다이버전스에서 처리하기
    
        #final_high_point['RSI_decreasing'] = final_high_point['RSI'] <= final_high_point['RSI'].shift()
        
        return self.__get_increase_sections(self.target_column, self.resource_df, self.minimum_interval,self.tolerate_interval)
        
        
    
    def __get_increase_sections(self,target_column_name, resource_df, minimum_interval, tolerate_interval):
        current_true_count = 0
        current_false_count = 0
        
        validated_index_list = [] #유효한 인덱스만 담는 리스트
        validated_group_list = [] #유효한 인덱스를 묶은 리스트
        
        start_index = 0
        end_index = 1
        
        for idx, i in enumerate(range(len(resource_df))):
            
            #첫번째는 무조건 False이므로 무시
            if (i==0):
                continue
        
            
            current_vaidate_count = 0
            current_invalidate_count = 0
            
            start_index = None
            end_index = None
            
            # 1. 일단 현재 값이 True 여야하고 
            # 2. 이미 들어 있는 validated_index_list 에 값이 tolerate_interval 보다 많으면 안된다.
            # 3. 그리고 minimun_interval 값을 무난하게 초과했는지 봐야한다.
            
            #현재 인덱스 값이 True 인지 확인
            current_status = resource_df[i:i+1][target_column_name].iloc[0]
            
            
            #현재의 인덱스 숫자 획득
            current_row_index = resource_df[i:i+1].index[0]
            current_row_timestamp = resource_df[i:i+1]['timestamp_kst'].iloc[0]
            #print(f"[{current_true_count},{current_false_count}]{current_status}, {current_row_index}, {current_row_timestamp}, {validated_index_list}")
            
            #현재 포인터리스트의 상태를 확인
            
            if(current_status==False):  
                #조건에 맞을 경우는 list에 인덱스 정보를 넣음
                if(current_false_count==tolerate_interval): #기준 숫자를 초과한 연속 하락 발생
            
                    #만약 이미 true 가 기준 갯수 이상이었다면 현재의인덱스가 마지막 인덱스다.
                    if(current_true_count>= (minimum_interval)):
                        
                        #현재 인덱스를 리스트에 넣음
                        validated_index_list.append(current_row_index)
                        
                        #마무리 하기 위해 현재의 list 를 group에 추가
                        
                        #print(f"조건에 맞는 리스트 추가 : {validated_index_list}")
                        
                        validated_group_list.append(validated_index_list)
                        
                        
                        
                                      
                        pass
                    else: #기준이 한번도 안나오고 실패 하면 아무것도 안하고 초기화
                        pass
                    
                    #초기화작업
                    validated_index_list = []
                    current_true_count = 0
                    current_false_count = 0
                    start_index = i+1 #새로운 [시작:] 을 부여
                    end_index = i+1 #새로운 [:끝] 을 부여
                else: #바로 직전보다는 하락이지만 추세는 아직 유지되고 있을 떄 ,
                    current_false_count += 1  #실패카운트를 올리고
                    #발생한 False의 인덱스를 넣고 마무리
                    validated_index_list.append(current_row_index)
                    
                    
            else: #일단 추세 유지라고 판단되는 경우
                #if(current_true_count==(minimum_interval-1)): #이번 상승으로 조건을 충족할 경우
                #    #일단 현재의 결과를 넣고
                #    validated_index_list.append(current_row_index)
                #    
                #    #그룹에 추가하고
                #    validated_group_list.append(validated_index_list)
                #    
                #    #초기화
                #    validated_index_list = []
                #    current_true_count = 0
                #    current_false_count = 0
                #else:
                #    #현재 True 긴 한데 아직 조건 충족이 안되었을 경우
                #    current_true_count += 1
                #    validated_index_list.append(current_row_index)
                
                #만약 초기화 된 이후 첫번째 validated-index_list 라면, 바로 앞의 인덱스까지 append 해주어야 한다. 
                #현재가 true 라는 것은 이전것에서부터 true 라는 뜻이기 때문이다.
                if(len(validated_index_list)==0):
                    before_row_index = resource_df[i-1:i].index[0]
                    current_true_count += 1
                    validated_index_list.append(before_row_index)
                
                
                #일단 true 인경우는 그냥 전체 구간으로 인식하게 둘 경우는 위에 주석 처리
                current_true_count += 1
                validated_index_list.append(current_row_index)
                
                #추세 유지지만 마지막까지 다다른 경우는 최종 값으로 추가해줘야 함
                if(idx == (len(resource_df) -1 )):
                    if(len(validated_index_list) >= (minimum_interval)): #현재의 조건이 맞는 경우
                        #print(f"조건에 맞는 리스트 추가 : {validated_index_list}")
                        
                        validated_group_list.append(validated_index_list)
                
                    
        return validated_group_list
    
    
# 데이터 시각화 인터페이스
class Visualization(ABC):
    
    @abstractmethod
    def make_figure(self):
        pass
    
    @abstractmethod
    def visualize(self):
        pass
    
    
    @abstractmethod
    def add_trace(self, trace):
        pass
    #self.price_trace_list.append(trace)
    

# 구체적인 시각화 전략: 라인 그래프
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class BasicPriceWithRsiVisualization(Visualization):
    
    def __init__(self):
        self.price_trace_list = []
        self.rsi_trace_lsit = []
        
    def set_data(self, df, high_points_df, low_points_df):
        self.__df = df
        self.__high_points_df = high_points_df
        self.__low_points_df = low_points_df
    
    def make_figure(self):
        self.__get_base_figure()
    
    def visualize(self):
        self.__main_figure.show()
        
    def add_trace(self, trace):
        
        #가격 트레이스 추가
        self.price_trace_list.append(trace)
        
        #메인 피겨 갱신
        self.__main_figure = self.__draw_subplots(self.price_trace_list,[rsi_trace])
        
    def get_figure(self):
        return self.__main_figure
        
        
    def __get_base_figure(self):
        main_trace = go.Candlestick(
            x=list(self.__df['timestamp_kst']),
            open=list(self.__df['open']),
            high=list(self.__df['high']),
            low=list(self.__df['low']),
            close=list(self.__df['close']),
        )
        
        #두번째 고가 점 트레이스 만들기
        high_point_trace = go.Scatter(y=list(self.__high_points_df['high_for_graph']), x=list(self.__high_points_df['timestamp_kst']), 
        marker=dict(
        color='rgba(255, 0, 255, 1)',  # 점의 색상 설정 (RGBa 형식)
        size=5,  # 점의 크기 설정
        ),
        mode='markers', name='고점 그래프')
        
        #세번째 저가 점 트레이스 만들기
        low_point_trace = go.Scatter(y=list(self.__low_points_df['low_for_graph']), x=list(self.__low_points_df['timestamp_kst']), 
        marker=dict(
        color='rgba(0, 0, 0, 1)',  # 점의 색상 설정 (RGBa 형식)
        size=5,  # 점의 크기 설정
        ),
        mode='markers', name='저점 그래프')
        
        ma_20_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['ma_20'], mode='lines', name='MA_20', line=dict(color='blue'))

        ma_60_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['ma_60'], mode='lines', name='MA_60', line=dict(color='black'))
        
        rsi_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['RSI'], mode='lines', name='RSI', line=dict(color='blue'))
        
        self.price_trace_list.append(main_trace)
        self.price_trace_list.append(high_point_trace)
        self.price_trace_list.append(low_point_trace)
        self.price_trace_list.append(ma_20_trace)
        self.price_trace_list.append(ma_60_trace)
        
        main_figure = self.__draw_subplots(self.price_trace_list,[rsi_trace])
        
        self.__main_figure = main_figure
   
    
    def __draw_subplots(self, price_trace_list, rsi_trace_list):
        
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        row_heights=[0.7, 0.3],  # 위쪽(가격) 70%, 아래쪽(RSI) 30%
                        subplot_titles=("가격 차트", "RSI (14)"))
        
        fig.update_layout(
        title='Candlestick Chart',
        xaxis_title='Date',
        yaxis_title='Price',
        xaxis_rangeslider_visible=False,
        width=1000,  # 그래프 너비 설정
        height=800   # 그래프 높이 설정
        )
        
        #trace_list1를 전부 담음
        for i in price_trace_list:
            fig.add_trace(i, row=1, col=1)
        
        for i in rsi_trace_list:
            fig.add_trace(i, row=2, col=1)
            fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought (70)", row=2, col=1)
            fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold (30)", row=2, col=1)
            
            
        return fig

        

# 알림 전략 인터페이스
class NotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message: str):
        pass

# 구체적인 알림 전략: 문자 메시지 전송
class SMSNotification(NotificationStrategy):
    def send_notification(self, message: str):
        print(f"문자 메시지 전송: {message}")

        
class PatternDetector(ABC):
    
    @abstractmethod
    def load(self,base_delta, from_date):
        pass
    
    @abstractmethod
    def add_sub_indicator(self,indicator_instance_list):
        pass
    
    @abstractmethod
    def execute(self):
        pass
        
        
# Bitmex 클래스 (컨텍스트)
class Bitmex(PatternDetector):
    #def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
    #             visualizer: VisualizationStrategy, notifier: NotificationStrategy):
    #    self.data_loader = data_loader
    #    self.processor = processor
    #    self.visualizer = visualizer
    #    self.notifier = notifier
    #    self.data = None
        
    def __init__(self):
        pass
        
        
    def set_loader(self,data_loader : DataLoaderStrategy):
        self.data_loader = data_loader
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    def set_processor(self, processor: DataProcessingStrategy):
        self.processor = processor
        
    def add_sub_indicator(self,indicator_instance_list):
        '''외부에서 주입받은 DataProcessingStrategy 중, 보조지표 추가하는 작업으로 정의된 클래스를 수행시킨다.
        보조지표의 추가이기  때문에 self.data 에 지속 쌓이는 데이터이다      
        '''
        for one_indicator in indicator_instance_list:
            #print("데이터확인")
            #print(self.data)
            self.data = one_indicator.process_data(self.data)
    
    def generate_high_low_data(self, high_point_processor, low_point_processor):
        self.high_point_df = high_point_processor.process_data(self.data)
        self.low_point_df = low_point_processor.process_data(self.data)
    
    def get_key_points(self, data):
        '''고점을 찾거나 다이버전스를 찾는등의 주요 포인트를 찾을 때 활용 다만 그 결과를 리턴받을 때만 쓴다.'''
        return self.processor.process_data(data)
    
            
        
    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

# 사용 예시
#if __name__ == "__main__":
#    bitmex = Bitmex(CSVDataLoader(), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#    bitmex.load()


In [9]:
import pyupbit
df = pyupbit.get_ohlcv('KRW-AUCTION', interval="minute60", count=2000)

In [10]:
df

,open,high,low,close,volume,value
2025-02-08 22:00:00,13050.0,13220.0,13030.0,13130.0,969.069321,1.268923e+07
2025-02-08 23:00:00,13120.0,13330.0,13120.0,13280.0,535.489245,7.091162e+06
2025-02-09 00:00:00,13280.0,13340.0,13140.0,13170.0,1873.506068,2.480500e+07
2025-02-09 01:00:00,13170.0,13310.0,13170.0,13310.0,1777.686263,2.355074e+07
2025-02-09 02:00:00,13330.0,13510.0,13330.0,13480.0,964.732853,1.294013e+07
...,...,...,...,...,...,...
2025-05-03 05:00:00,16830.0,16930.0,16820.0,16870.0,4103.578664,6.930453e+07
2025-05-03 06:00:00,16870.0,16920.0,16790.0,16820.0,2482.275269,4.184019e+07
2025-05-03 07:00:00,16820.0,16890.0,16730.0,16890.0,4654.084373,7.832003e+07
2025-05-03 08:00:00,16860.0,16980.0,16860.0,16960.0,6576.791355,1.113221e+08


In [11]:
coin_name = "KRW-AUCTION"
inta = "minute60"

class Test:
    def load_data(self):
        df = pyupbit.get_ohlcv(coin_name, interval=inta, count=2000)
        print(df)
        
t = Test()
t.load_data()

                        open     high      low    close       volume  \
2025-02-08 22:00:00  13050.0  13220.0  13030.0  13130.0   969.069321   
2025-02-08 23:00:00  13120.0  13330.0  13120.0  13280.0   535.489245   
2025-02-09 00:00:00  13280.0  13340.0  13140.0  13170.0  1873.506068   
2025-02-09 01:00:00  13170.0  13310.0  13170.0  13310.0  1777.686263   
2025-02-09 02:00:00  13330.0  13510.0  13330.0  13480.0   964.732853   
...                      ...      ...      ...      ...          ...   
2025-05-03 05:00:00  16830.0  16930.0  16820.0  16870.0  4103.578664   
2025-05-03 06:00:00  16870.0  16920.0  16790.0  16820.0  2482.275269   
2025-05-03 07:00:00  16820.0  16890.0  16730.0  16890.0  4654.084373   
2025-05-03 08:00:00  16860.0  16980.0  16860.0  16960.0  6576.791355   
2025-05-03 09:00:00  16930.0  17010.0  16890.0  16900.0  8371.772142   

                            value  
2025-02-08 22:00:00  1.268923e+07  
2025-02-08 23:00:00  7.091162e+06  
2025-02-09 00:00:00  2.4805

In [12]:
# 구체적인 데이터 로드 전략: CSV 로드
class UpbitRealtimeDataLoader(DataLoaderStrategy):
    
    def __init__(self, coin_id, base_delta, count:int = 1000):
        self.coin_id = coin_id
        self.base_delta = base_delta
        self.count = count
        
        
    
    def load_data(self):
        print(f"업비트에서 {self.coin_id} 가격을 최신부터{self.base_delta} 간격으로 {self.count}개 로드 합니다.")
        import pyupbit
        df = pyupbit.get_ohlcv(self.coin_id, interval=self.base_delta, count=self.count)
        
        #인덱스를 컬럼으로 가져오기
        df = df.reset_index()
        df = df.rename(columns={'index': 'timestamp_kst'})
        
        df = df[['timestamp_kst','open','low','high','close']]
        
        return df
    


# Bitmex 클래스 (컨텍스트)
class Upbit(PatternDetector):
    #def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
    #             visualizer: VisualizationStrategy, notifier: NotificationStrategy):
    #    self.data_loader = data_loader
    #    self.processor = processor
    #    self.visualizer = visualizer
    #    self.notifier = notifier
    #    self.data = None
        
    def __init__(self):
        pass
        
        
    def set_loader(self,data_loader : DataLoaderStrategy):
        self.data_loader = data_loader
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    def set_processor(self, processor: DataProcessingStrategy):
        self.processor = processor
        
    def add_sub_indicator(self,indicator_instance_list):
        '''외부에서 주입받은 DataProcessingStrategy 중, 보조지표 추가하는 작업으로 정의된 클래스를 수행시킨다.
        보조지표의 추가이기  때문에 self.data 에 지속 쌓이는 데이터이다      
        '''
        for one_indicator in indicator_instance_list:
            #print("데이터확인")
            #print(self.data)
            self.data = one_indicator.process_data(self.data)
    
    def generate_high_low_data(self, high_point_processor, low_point_processor):
        self.high_point_df = high_point_processor.process_data(self.data)
        self.low_point_df = low_point_processor.process_data(self.data)
    
    def get_key_points(self, data):
        '''고점을 찾거나 다이버전스를 찾는등의 주요 포인트를 찾을 때 활용 다만 그 결과를 리턴받을 때만 쓴다.'''
        return self.processor.process_data(data)

    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

In [36]:
def find_overlapping_intervals(a, b):
    result = []
    index_dict = {} #인덱스에 해당하는 high low 구간을 기록
    
    for a_start, a_end in a:
        for b_start, b_end in b:
            # 겹치는 부분 확인
            overlap_start = max(a_start, b_start)
            overlap_end = min(a_end, b_end)
            
            if overlap_start <= overlap_end:  # 유효한 겹침 구간
                result.append([overlap_start, overlap_end])
                #만약 겹침 구간이 등록되었다면 그때 인덱스에서 활용된 high / low 인덱스리스트를 기록
                index_dict[overlap_end] = {}
                index_dict[overlap_end]['high_section'] = [a_start,a_end]
                index_dict[overlap_end]['low_section'] = [b_start,b_end]
                
    if(len(result)!=0):
        return result, index_dict
    else:
        return result, index_dict
    #return result

def add_vrect_to_main_figure(main_figure, df, color, group_list):
    for one_list in group_list:
        #df순서대로 조회하면서 
        start_end_df = df.loc[[one_list[0],one_list[-1]]]
        start_date = start_end_df['timestamp_kst'].iloc[0]
        end_date = start_end_df['timestamp_kst'].iloc[-1]
        
        main_figure.add_vrect(
        x0=start_date, x1=end_date,  # 색칠할 x 구간
        fillcolor=color, opacity=0.3,  # 색상 및 투명도
        layer="below",  # 그래프 아래 배치
        line_width=0  # 테두리 제거
        )
    
    return main_figure

def generate_jpg_file_name(coin_name):
    import datetime
    post_fixt = datetime.datetime.now().strftime('%Y%m%d_%H.jpg')
    return coin_name+'_'+post_fixt
    

In [ ]:
#4시간봉 또는 일봉의 고점이 그 전에 올라갔는지 판단하는 로직 추가

#60분일 때 
interval_base = "minute240"
load_count = 400
score_band_list = [42] #일주일에 1번 채점
threshold = 0.85
indecreasing_count_set=[3,0] #3번 계속 상승구간만 체크하고 허용을 0번함

base_date_dict = {} #코인 별 변곡 기준이 되는 시간을 담은 사전

count = 0
#240분일 떄
#interval_base = "minute240"
#load_count = 200
#score_band_list = [30] #5일중 제일 높은 구간 채점
#threshold = 0.85
#indecreasing_count_set=[3,1]


## 업비트 전체 뒤져보기
tickers = pyupbit.get_tickers('KRW')

#코인별 확인
for idx, one_coin in enumerate(tickers):
    
    #if(one_coin not in ['KRW-SNT']):
    #    continue
     
    
    #if(one_coin not in target_coin_name_list):
    #    continue
    
    
    print(f"{one_coin} 시작")
    
    #RAW 데이터 로드
    upbit = Upbit()
    upbit.set_loader(UpbitRealtimeDataLoader(one_coin,interval_base,load_count))
    upbit.load()
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-27 00:00:00']
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-26 23:00:00']
    
    #지표 추가
    indicator_list = [MovingAverageProcessing(), RSIProcessing(),HighPointScoringProcessing(score_band_list),LowPointScoringProcessing(score_band_list)]
    upbit.add_sub_indicator(indicator_list)
    
    #고점, 저점 찾기
    upbit.generate_high_low_data(GetHighPoints(upbit.data.loc[upbit.data['high_score']!=0], 'high_score', threshold), 
                                GetLowPoints(upbit.data.loc[upbit.data['low_score']!=0], 'low_score', threshold)
                                )
    
    #고.저점 상승 추세구간 획득
    upbit.set_processor(GetTrendSections('high','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    upbit.set_processor(GetTrendSections('high','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    
    upbit.set_processor(GetTrendSections('low','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    upbit.set_processor(GetTrendSections('low','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    
    
    #저점이 하락하다가 상승하는 것만 색출함
    if(list(upbit.low_point_df[-3:]['increasing'])!=[False, False, True]):
        continue
    
    
    
    #저점이 올라간 마지막 날을 기준일로 dict 에 넣음
    base_date_dict[one_coin]=upbit.low_point_df[-1:]['timestamp_kst']
    
    #출력
    draw_instance = BasicPriceWithRsiVisualization()
    draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
    draw_instance.make_figure()
    
    figure = draw_instance.get_figure()
    
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_low_increasing_trend_section_upbit) #중첩된 구간만 그리기
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_point_group_start_end_upbit) #고점 그래프는 빨간색
    #add_vrect_to_main_figure(figure, upbit.data, 'blue', low_point_group_start_end_upbit) #고점 그래프는 파란색
    #draw_instance.visualize()
    
    #count += 1
    
    #if(count==1):
    #    break

In [17]:
base_date_dict

{}

In [18]:
len(base_date_dict.keys())
len(tickers)

169

쓸만한설정값
#60분일 때 
interval_base = "minute60"
load_count = 500
score_band_list = [12] #12시간에 한번씩 채점
threshold = 0.90
indecreasing_count_set=[3,0] #3번 계속 상승구간만 체크하고 허용을 0번함

In [ ]:


#60분일 때 
interval_base = "minute60"
load_count = 240
score_band_list = [12] #12시간에 한번씩 채점
threshold = 0.90
indecreasing_count_set=[3,0] #3번 계속 상승구간만 체크하고 허용을 0번함

figure_to_jpg_dict = {} #저장한 이미지와 그 경로를 가진 dict 를 생성하고 아래카카오에서 이 정보를 기반으로 메시지를 전송함

#240분일 떄
#interval_base = "minute240"
#load_count = 200
#score_band_list = [30] #5일중 제일 높은 구간 채점
#threshold = 0.85
#indecreasing_count_set=[3,1]

target_coin_name_list = []

## 업비트 전체 뒤져보기
tickers = pyupbit.get_tickers('KRW')

#코인별 확인
for idx, one_coin in enumerate(tickers):
    
    #if one_coin not in base_date_dict.keys(): #최근 변곡점이 아닌 애들이라면 아예 하지 않음
    #    continue
    
    #if(one_coin not in ['KRW-NEAR']):
    #    continue
    
    print(f"{one_coin} 시작")
    
    #RAW 데이터 로드
    upbit = Upbit()
    upbit.set_loader(UpbitRealtimeDataLoader(one_coin,interval_base,load_count))
    upbit.load()
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-27 00:00:00']
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-26 23:00:00']
    
    #지표 추가
    indicator_list = [MovingAverageProcessing(), RSIProcessing(),HighPointScoringProcessing(score_band_list),LowPointScoringProcessing(score_band_list)]
    upbit.add_sub_indicator(indicator_list)
    
    #고점, 저점 찾기
    upbit.generate_high_low_data(GetHighPoints(upbit.data.loc[upbit.data['high_score']!=0], 'high_score', threshold), 
                                GetLowPoints(upbit.data.loc[upbit.data['low_score']!=0], 'low_score', threshold)
                                )
    
    #고.저점 상승 추세구간 획득
    upbit.set_processor(GetTrendSections('high','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    upbit.set_processor(GetTrendSections('high','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    
    upbit.set_processor(GetTrendSections('low','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    upbit.set_processor(GetTrendSections('low','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    
    #저점고점 함께 상승하는 구간 찾기
    high_point_group_start_end_upbit = []
    for i in high_point_increasing_trend_section_upbit:
        high_point_group_start_end_upbit.append([i[0],i[-1]])
    
    
    low_point_group_start_end_upbit = []
    for i in low_point_increasing_trend_section_upbit:
        low_point_group_start_end_upbit.append([i[0],i[-1]])

    
    #고점은 낮아지는 구간 찾기
    #high_point_group_start_end_upbit = []
    #for i in high_point_decreasing_trend_section_upbit:
    #    high_point_group_start_end_upbit.append([i[0],i[-1]])
    
    
    
    high_low_increasing_trend_section_upbit, high_low_list_dict = find_overlapping_intervals(high_point_group_start_end_upbit, low_point_group_start_end_upbit)
    
    #중첩되었을 떄의 고점 리스트는
    
    #중첩 구간이 가장 최근인지 확인
    if(len(high_low_increasing_trend_section_upbit) != 0 ):
        #중첩된 구간의 마지막 인덱스를 찾고
        latest_overlap_index = high_low_increasing_trend_section_upbit[-1][-1] #가장 가까운 구간의 인덱스 값을 찾고
        #중첩이 되었을 떄 활용되었던 고점의 마지막 인덱스를 획득하고
        latest_high_index = high_low_list_dict[latest_overlap_index]['high_section'][-1]
        
        #중첩이 되었을 때 활용되었던 저점의 마지막 인덱스를 획득
        latest_low_index = high_low_list_dict[latest_overlap_index]['low_section'][-1]
        
        print(f"기준이 되는 고점 날짜 ; {upbit.data.iloc[[latest_high_index]]['timestamp_kst'].iloc[0]}")
        print(f"기준이 되는 저점 날짜 ; {upbit.data.iloc[[latest_low_index]]['timestamp_kst'].iloc[0]}")
        
        latest_timestamp_high = upbit.data.iloc[[latest_high_index]]['timestamp_kst'].iloc[0] #마지막 고점구간의 끝 날짜 확인
        latest_timestamp_low = upbit.data.iloc[[latest_low_index]]['timestamp_kst'].iloc[0] #마지막 저점구간의 끝 날짜 확인
        check_df_high = upbit.data.loc[upbit.data['timestamp_kst'] >= latest_timestamp_high] #그 날짜보다 높은 값이 있는지 확인
        check_df_low = upbit.data.loc[upbit.data['timestamp_kst'] >= latest_timestamp_low] #그 날짜보다 높은 값이 있는지 확인
        
        print(f"기준 고점 이후 몇시간 지났나 : {len(check_df_high)}")
        print(f"기준 저점 이후 몇시간 지났나 : {len(check_df_low)}")
        
        #print(check_df)
        #print(len(check_df))
        if(len(check_df_high)<11 or len(check_df_low)<11): #1개만 발견됐다면 이건 바로 지금 유의미한 자리인 것
            pass
        else:
            continue
    else: #중첩 구간이 없는건 일단 무시
        continue
        
    #일단 주목해야할 코인들의 리스트를 받은
    target_coin_name_list.append(one_coin)
    
    
    #출력
    draw_instance = BasicPriceWithRsiVisualization()
    draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
    draw_instance.make_figure()
    
    figure = draw_instance.get_figure()
    
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_low_increasing_trend_section_upbit) #중첩된 구간만 그리기
    add_vrect_to_main_figure(figure, upbit.data, 'red', high_point_group_start_end_upbit) #고점 그래프는 빨간색
    add_vrect_to_main_figure(figure, upbit.data, 'blue', low_point_group_start_end_upbit) #고점 그래프는 파란색
    draw_instance.visualize()
    
    #파일이름으로 저장함
    
    jpg_file_name = JPG_DIRECTORY+'/'+generate_jpg_file_name(one_coin)
    print(f"파일 저장 :{jpg_file_name}")
    figure.write_image(jpg_file_name, format="jpg")
    print("완료")
    
    
    break
    
    #if(idx==2):
    #    break
    
    
    


In [35]:
generate_jpg_file_name('KRW_XRP')

'KRW_XRP_20250503_09.jpg'

In [ ]:
target_coin_name_list

In [26]:
import plotly.express as px

figure.write_image("c://Test/plot.jpg", format="jpg")

In [ ]:
#4시간봉 또는 일봉의 고점이 그 전에 올라갔는지 판단하는 로직 추가

#60분일 때 
interval_base = "minute240"
load_count = 400
score_band_list = [42] #일주일에 1번 채점
threshold = 0.85
indecreasing_count_set=[3,0] #3번 계속 상승구간만 체크하고 허용을 0번함

base_date_dict = {} #코인 별 변곡 기준이 되는 시간을 담은 사전


#240분일 떄
#interval_base = "minute240"
#load_count = 200
#score_band_list = [30] #5일중 제일 높은 구간 채점
#threshold = 0.85
#indecreasing_count_set=[3,1]


## 업비트 전체 뒤져보기
tickers = pyupbit.get_tickers('KRW')

#코인별 확인
for idx, one_coin in enumerate(tickers):
    
    #if(one_coin not in ['KRW-SNT']):
    #    continue
     
    
    if(one_coin not in target_coin_name_list):
        continue
    
    
    print(f"{one_coin} 시작")
    
    #RAW 데이터 로드
    upbit = Upbit()
    upbit.set_loader(UpbitRealtimeDataLoader(one_coin,interval_base,load_count))
    upbit.load()
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-27 00:00:00']
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-26 23:00:00']
    
    #지표 추가
    indicator_list = [MovingAverageProcessing(), RSIProcessing(),HighPointScoringProcessing(score_band_list),LowPointScoringProcessing(score_band_list)]
    upbit.add_sub_indicator(indicator_list)
    
    #고점, 저점 찾기
    upbit.generate_high_low_data(GetHighPoints(upbit.data.loc[upbit.data['high_score']!=0], 'high_score', threshold), 
                                GetLowPoints(upbit.data.loc[upbit.data['low_score']!=0], 'low_score', threshold)
                                )
    
    #고.저점 상승 추세구간 획득
    upbit.set_processor(GetTrendSections('high','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    upbit.set_processor(GetTrendSections('high','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    
    upbit.set_processor(GetTrendSections('low','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    upbit.set_processor(GetTrendSections('low','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    
    
    #저점이 하락하다가 상승하는 것만 색출함
    if(list(upbit.low_point_df[-3:]['increasing'])!=[False, False, True]):
        continue
    
    #저점이 올라간 마지막 날을 기준일로 dict 에 넣음
    base_date_dict[one_coin]=upbit.low_point_df[-1:]['timestamp_kst']
    
    #출력
    draw_instance = BasicPriceWithRsiVisualization()
    draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
    draw_instance.make_figure()
    
    figure = draw_instance.get_figure()
    
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_low_increasing_trend_section_upbit) #중첩된 구간만 그리기
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_point_group_start_end_upbit) #고점 그래프는 빨간색
    #add_vrect_to_main_figure(figure, upbit.data, 'blue', low_point_group_start_end_upbit) #고점 그래프는 파란색
    #draw_instance.visualize()
    
    if(idx==5):
        break
    

In [ ]:
base_date_dict

In [ ]:
upbit.low_point_df[-1:]['timestamp_kst'] == '2025-04-18 09:00:00'

In [ ]:
list(upbit.low_point_df[-3:]['increasing'])==[False, True, False]

In [ ]:
upbit.low_point_df

In [ ]:
high_low_list_dict

In [ ]:
high_low_increasing_trend_section_upbit

In [ ]:
upbit.data

In [ ]:

#중첩된 구간이 가장 최근인지 확인
high_low_increasing_trend_section_upbit[-1][-1] #가장 최근 구간 중에 제일 끝 지점만 확인



In [ ]:
upbit.data.iloc[[103]]['timestamp_kst'].iloc[0]

In [ ]:
upbit.data.loc[upbit.data['timestamp_kst'] >= '2025-03-23 20:00:00']

In [ ]:
upbit.data.iloc[[62,103]]

In [ ]:
upbit = Upbit()
upbit.set_loader(UpbitRealtimeDataLoader('KRW-MLK',interval_base,load_count))
upbit.load()

upbit.data[:196]

In [ ]:
upbit.high_point_df

In [ ]:
upbit.low_point_df

In [ ]:
high_point_increasing_trend_section_upbit

In [ ]:
upbit.set_processor(GetTrendSections('high','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
high_point_increasing_trend_section_upbit

In [ ]:
upbit = Upbit()
upbit.set_loader(UpbitRealtimeDataLoader('KRW-AUCTION',"minute60",2000))

upbit.load()

In [ ]:
upbit.data

In [ ]:
indicator_list = [MovingAverageProcessing(), RSIProcessing(),HighPointScoringProcessing([12]),LowPointScoringProcessing([12])]
upbit.add_sub_indicator(indicator_list)

In [ ]:
upbit.generate_high_low_data(GetHighPoints(upbit.data.loc[upbit.data['high_score']!=0], 'high_score', 0.85), 
                                GetLowPoints(upbit.data.loc[upbit.data['low_score']!=0], 'low_score', 0.85)
                                )

In [ ]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
draw_instance.make_figure()
draw_instance.visualize()

In [ ]:
upbit.set_processor(GetTrendSections('high','increasing',3,0))
high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
upbit.set_processor(GetTrendSections('high','decreasing',3,0))
high_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)

upbit.set_processor(GetTrendSections('low','increasing',3,0))
low_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
upbit.set_processor(GetTrendSections('low','decreasing',3,0))
low_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)

In [ ]:
high_point_increasing_trend_section_upbit

In [ ]:
low_point_increasing_trend_section_upbit

In [ ]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
draw_instance.make_figure()
figure = draw_instance.get_figure()
add_vrect_to_main_figure(figure, upbit.data, 'red', high_point_increasing_trend_section_upbit)
#add_vrect_to_main_figure(figure, upbit.data, 'blue', low_point_increasing_trend_section_upbit)

In [ ]:
high_point_group_start_end_upbit = []
for i in high_point_increasing_trend_section_upbit:
    high_point_group_start_end_upbit.append([i[0],i[-1]])
high_point_group_start_end_upbit

low_point_group_start_end_upbit = []
for i in low_point_increasing_trend_section_upbit:
    low_point_group_start_end_upbit.append([i[0],i[-1]])
low_point_group_start_end_upbit



high_low_increasing_trend_section_upbit = find_overlapping_intervals(high_point_group_start_end_upbit, low_point_group_start_end_upbit)
high_low_increasing_trend_section_upbit

In [ ]:
def add_vrect_to_main_figure(main_figure, df, color, group_list):
    for one_list in group_list:
        #df순서대로 조회하면서 
        start_end_df = df.loc[[one_list[0],one_list[-1]]]
        start_date = start_end_df['timestamp_kst'].iloc[0]
        end_date = start_end_df['timestamp_kst'].iloc[-1]
        
        main_figure.add_vrect(
        x0=start_date, x1=end_date,  # 색칠할 x 구간
        fillcolor=color, opacity=0.3,  # 색상 및 투명도
        layer="below",  # 그래프 아래 배치
        line_width=0  # 테두리 제거
        )
    
    return main_figure

draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
draw_instance.make_figure()

figure = draw_instance.get_figure()
add_vrect_to_main_figure(figure, upbit.data, 'red', high_low_increasing_trend_section_upbit)

In [ ]:
bitmex = Bitmex()

import platform
if('Intel64 Family 6 Model 126 Stepping 5, GenuineIntel'==platform.processor()):
    bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
else:
    bitmex.set_loader(BitmexCSVDataLoader(15,'2023-12-31 15:01:00'))
bitmex.load()

In [ ]:
bitmex.data

In [ ]:
indicator_list = [MovingAverageProcessing(), RSIProcessing()]
bitmex.add_sub_indicator(indicator_list)

In [ ]:
indicator_list_more = [HighPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

indicator_list_more = [LowPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

In [ ]:
bitmex.data

In [ ]:
bitmex.generate_high_low_data(GetHighPoints(bitmex.data.loc[bitmex.data['high_score']!=0], 'high_score', 0.90), 
                                GetLowPoints(bitmex.data.loc[bitmex.data['low_score']!=0], 'low_score', 0.90)
                                )

In [ ]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(bitmex.data, bitmex.high_point_df, bitmex.low_point_df)
draw_instance.make_figure()
draw_instance.visualize()

In [ ]:
bitmex_60 = Bitmex()
#bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
bitmex_60.set_loader(BitmexCSVDataLoader(60,'2022-12-31 15:01:00'))
bitmex_60.load()

indicator_list_60 = [MovingAverageProcessing(), RSIProcessing()]
bitmex_60.add_sub_indicator(indicator_list_60)

indicator_list_more_60 = [HighPointScoringProcessing([48])]
bitmex_60.add_sub_indicator(indicator_list_more_60)

indicator_list_more_60 = [LowPointScoringProcessing([48])]
bitmex_60.add_sub_indicator(indicator_list_more_60)




In [ ]:
bitmex_60.generate_high_low_data(GetHighPoints(bitmex_60.data.loc[bitmex_60.data['high_score']!=0], 'high_score', 0.90), 
                                GetLowPoints(bitmex_60.data.loc[bitmex_60.data['low_score']!=0], 'low_score', 0.90)
                                )

In [ ]:
bitmex_60.high_point_df

In [ ]:
bitmex_60.low_point_df

In [ ]:
bitmex_60.data

In [ ]:

#bitmex_60.get_key_points(bitmex_60.high_point_df)

In [ ]:
bitmex_60.low_point_df

In [ ]:
bitmex_60.set_processor(GetTrendSections('high','increasing',5,1))
high_point_increasing_trend_section = bitmex_60.get_key_points(bitmex_60.high_point_df)
bitmex_60.set_processor(GetTrendSections('high','decreasing',5,1))
high_point_decreasing_trend_section = bitmex_60.get_key_points(bitmex_60.high_point_df)

bitmex_60.set_processor(GetTrendSections('low','increasing',5,1))
low_point_increasing_trend_section = bitmex_60.get_key_points(bitmex_60.low_point_df)
bitmex_60.set_processor(GetTrendSections('low','decreasing',5,1))
low_point_decreasing_trend_section = bitmex_60.get_key_points(bitmex_60.low_point_df)

In [ ]:
high_point_group_start_end = []
for i in high_point_increasing_trend_section:
    high_point_group_start_end.append([i[0],i[-1]])
high_point_group_start_end

low_point_group_start_end = []
for i in low_point_increasing_trend_section:
    low_point_group_start_end.append([i[0],i[-1]])
low_point_group_start_end

def find_overlapping_intervals(a, b):
    result = []
    
    for a_start, a_end in a:
        for b_start, b_end in b:
            # 겹치는 부분 확인
            overlap_start = max(a_start, b_start)
            overlap_end = min(a_end, b_end)
            
            if overlap_start <= overlap_end:  # 유효한 겹침 구간
                result.append([overlap_start, overlap_end])
    
    return result

high_low_increasing_trend_section = find_overlapping_intervals(high_point_group_start_end, low_point_group_start_end)
high_low_increasing_trend_section

In [ ]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(bitmex_60.data, bitmex_60.high_point_df, bitmex_60.low_point_df)
draw_instance.make_figure()
draw_instance.visualize()

In [ ]:


figure = draw_instance.get_figure()
add_vrect_to_main_figure(figure, bitmex_60.data, 'red', high_low_increasing_trend_section)

In [ ]:
import requests

REST_API_KEY = "44f0eb9ebc88a986c0855aae35a3790a" #하민용
REDIRECT_URI = "http://localhost:5000"
CODE = "TOaufAxUu_IEIj9M5IkEKaYAn8g06O5dSryrugb05ocdlImjcCeKYgAAAAQKDRtZAAABlf8Ktau2xj-RG-1vuA"  # 위에서 받은 인증 코드

url = "https://kauth.kakao.com/oauth/token"
data = {
    "grant_type": "authorization_code",
    "client_id": REST_API_KEY,
    "redirect_uri": REDIRECT_URI,
    "code": CODE,
}

response = requests.post(url, data=data)
tokens = response.json()
print(tokens)

In [ ]:
#내 전용

import requests

REST_API_KEY = "20ba59c1dc1dcdca7f2cdd0747686fe6"
REDIRECT_URI = "https://localhost:3000"
CODE = "EcGsGiXAUBAD28K0I_4BSU-wez8b4JPHfPUVHJlakUrUw2cGyPImZwAAAAQKDQgeAAABljd0xHlUdd9ffL_GXA"  # 위에서 받은 인증 코드


url = "https://kauth.kakao.com/oauth/token"
data = {
    "grant_type": "authorization_code",
    "client_id": REST_API_KEY,
    "redirect_uri": REDIRECT_URI,
    "code": CODE,
}

response = requests.post(url, data=data)
tokens = response.json()
print(tokens)

In [ ]:
#code 를 얻는 과정
import urllib.parse

base_url = "https://kauth.kakao.com/oauth/authorize"

params = {
    "response_type": "code",
    "client_id": REST_API_KEY,
    "redirect_uri": REDIRECT_URI,
    "scope": "friends"
}

url = f"{base_url}?{urllib.parse.urlencode(params)}"
print("Visit this URL to authorize:", url)

In [ ]:
#하민거

import requests


ACCESS_TOKEN = 'OpGolhiuH8oTD40MSEp5h4d0PYR2bg-1AAAAAQoXFO4AAAGV_wMYPHLErHmNOyL0'
ACCESS_TOKEN = 'uXXoD3O2JLtAd4SnJPwWcF8mLlBJpOm5AAAAAQoXNVcAAAGV_wr_jnLErHmNOyL0'

url = "https://kapi.kakao.com/v2/api/talk/memo/default/send"
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
data = {
    "template_object": '{"object_type":"text","text":"알람 메시지 도착!","link":{"sample":"sample"}}'
}

response = requests.post(url, headers=headers, data=data)

# 결과 출력
if response.status_code == 200:
    print("✅ 메시지 전송 성공!")
else:
    print(f"❌ 메시지 전송 실패: {response.json()}")

In [ ]:
#내거

import requests


ACCESS_TOKEN = 'BteYRO1PEm4_lEXXxrALD7uNqFXYPefPAAAAAQoXFmIAAAGWN3T3p3LErHmNOyL0'

url = "https://kapi.kakao.com/v2/api/talk/memo/default/send"
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
data = {
    "template_object": '{"object_type":"text","text":"알람 메시지 도착!","link":{"sample":"sample"}}'
}

response = requests.post(url, headers=headers, data=data)

# 결과 출력
if response.status_code == 200:
    print("✅ 메시지 전송 성공!")
else:
    print(f"❌ 메시지 전송 실패: {response.json()}")

In [ ]:
response

In [ ]:
#친구목록 불러오기 (테스트계정이라도 서로 친구 등록이 되어 있어야 뜬다.)
import requests


url = "https://kapi.kakao.com/v1/api/talk/friends"

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

response = requests.get(url, headers=headers)

friends = response.json()
print(friends)

In [ ]:
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

files = {
    "file": open("c://Temp/test.jpg", "rb")
}

res = requests.post(
    "https://kapi.kakao.com/v2/api/talk/message/image/upload",
    headers=headers,
    files=files
)

upload_result = res.json()
print(upload_result)

In [ ]:
upload_result['infos']['original']

In [ ]:
import json

template = {
    "object_type": "feed",
    "content": {
        "title": "업로드된 이미지 포함 메시지",
        "description": "로컬 이미지 업로드 후 전송",
        "image_url": upload_result['infos']['original']["url"],
        "image_width": upload_result['infos']['original']["width"],
        "image_height": upload_result['infos']['original']["height"],
        "link": {
            "web_url": "https://www.kakao.com",
            "mobile_web_url": "https://www.kakao.com"
        }
    }
}

data = {
    "template_object": json.dumps(template)
}

res = requests.post(
    "https://kapi.kakao.com/v2/api/talk/memo/default/send",
    headers=headers,
    data=data
)

print(res.status_code)
print(res.json())